# Credit Default Prediction — Part 4: Business Interpretation

## The Real Question

A model with AUC = 0.87 is technically strong. But the business question is not *"how good is the model?"* — it is:

> **At what approval threshold does the lender maximize risk-adjusted return?**

This notebook answers that question by sweeping across decision thresholds and quantifying the business impact of each — in terms of approval rate, expected default rate, and estimated loss. This is how a risk team at a consumer lender actually uses a model.

**Intended audience:** Product, Risk, and Strategy stakeholders — not just data scientists.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

plt.style.use('seaborn-v0_8-whitegrid')

y_test  = pd.read_csv('../data/y_test.csv').squeeze()
proba   = joblib.load('../outputs/test_probabilities.pkl')

# Assume average loan size of $1,500 (representative BNPL)
AVG_LOAN = 1500
# Assume ~40% recovery on defaulted loans
RECOVERY_RATE = 0.40
LOSS_GIVEN_DEFAULT = AVG_LOAN * (1 - RECOVERY_RATE)  # $900 per default
# Assume 8% net interest margin on performing loans
REVENUE_PER_PERFORMING = AVG_LOAN * 0.08  # $120

## 1. Threshold Sweep: Approval Rate vs. Default Rate

In [ ]:
thresholds = np.arange(0.05, 0.55, 0.01)
rows = []

for t in thresholds:
    approved = proba <= t  # approve if predicted default prob is below threshold
    n_approved = approved.sum()
    n_total = len(proba)
    approval_rate = n_approved / n_total

    if n_approved == 0:
        continue

    actual_defaults_approved = y_test[approved].sum()
    default_rate_approved = actual_defaults_approved / n_approved
    n_performing = n_approved - actual_defaults_approved

    expected_revenue = n_performing * REVENUE_PER_PERFORMING
    expected_loss = actual_defaults_approved * LOSS_GIVEN_DEFAULT
    net_value = expected_revenue - expected_loss

    rows.append({
        'threshold': round(t, 2),
        'approval_rate': approval_rate,
        'default_rate_approved': default_rate_approved,
        'n_approved': n_approved,
        'net_value': net_value,
        'expected_loss': expected_loss
    })

sweep = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Approval rate vs default rate
axes[0].plot(sweep['approval_rate'] * 100, sweep['default_rate_approved'] * 100,
             color='#e74c3c', linewidth=2)
axes[0].set_xlabel('Approval Rate (%)')
axes[0].set_ylabel('Default Rate Among Approved (%)')
axes[0].set_title('Approval Rate vs. Default Rate', fontsize=12)
for t_highlight in [0.10, 0.15, 0.20, 0.25]:
    row = sweep[sweep['threshold'] == t_highlight]
    if not row.empty:
        x = row['approval_rate'].values[0] * 100
        y = row['default_rate_approved'].values[0] * 100
        axes[0].annotate(f'p≤{t_highlight}', (x, y),
                         textcoords='offset points', xytext=(8, 0), fontsize=8)
        axes[0].plot(x, y, 'ko', markersize=5)

# Right: Net value by approval rate
axes[1].plot(sweep['approval_rate'] * 100, sweep['net_value'] / 1e6,
             color='#2ecc71', linewidth=2)
axes[1].set_xlabel('Approval Rate (%)')
axes[1].set_ylabel('Estimated Net Value ($M)')
axes[1].set_title('Net Value by Approval Rate', fontsize=12)
best = sweep.loc[sweep['net_value'].idxmax()]
axes[1].axvline(best['approval_rate'] * 100, color='#e74c3c', linestyle='--',
                label=f"Optimal: {best['approval_rate']:.0%} approval\n(threshold={best['threshold']})")
axes[1].legend(fontsize=9)

plt.suptitle('Approval Threshold Analysis — Business Impact', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Decision Table: Three Risk Appetite Scenarios

In [ ]:
scenarios = {
    'Conservative (p ≤ 10%)': 0.10,
    'Moderate (p ≤ 15%)':     0.15,
    'Growth (p ≤ 20%)':       0.20
}

print('\n=== Risk Appetite Scenario Comparison ===\n')
print(f'{"Scenario":<28} {"Approval Rate":>14} {"Default Rate":>13} {"Est. Net Value":>15}')
print('-' * 75)

for label, t in scenarios.items():
    row = sweep[sweep['threshold'] == t]
    if row.empty:
        row = sweep.iloc[(sweep['threshold'] - t).abs().argsort()[:1]]
    print(f"{label:<28} "
          f"{row['approval_rate'].values[0]:>13.1%} "
          f"{row['default_rate_approved'].values[0]:>12.1%} "
          f"${row['net_value'].values[0]/1e6:>13.2f}M")

print('\nNote: Net value estimates use avg loan $1,500, 40% recovery, 8% NIM on performing loans.')
print('Optimal threshold is determined by the business risk appetite, not model performance alone.')

## 3. Fair Lending Check — Default Rate Parity by Age Group

Under ECOA and fair lending principles, a model must not systematically disadvantage protected classes. Age is a sensitive attribute. We check whether the model's approval rates and error rates are materially different across age groups.

In [ ]:
X_test_raw = pd.read_csv('../data/X_test.csv')
X_test_raw['predicted_proba'] = proba
X_test_raw['actual_default'] = y_test.values

THRESHOLD = 0.15
X_test_raw['approved'] = (X_test_raw['predicted_proba'] <= THRESHOLD).astype(int)

X_test_raw['age_group'] = pd.cut(X_test_raw['age'],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=['<25', '25-34', '35-44', '45-54', '55-64', '65+'])

fairness = X_test_raw.groupby('age_group').agg(
    approval_rate=('approved', 'mean'),
    default_rate_approved=('actual_default', lambda x: x[X_test_raw.loc[x.index, 'approved'] == 1].mean())
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title, color in zip(
    axes,
    ['approval_rate', 'default_rate_approved'],
    ['Approval Rate by Age Group', 'Default Rate (Approved) by Age Group'],
    ['#3498db', '#e74c3c']
):
    ax.bar(fairness['age_group'], fairness[col], color=color, edgecolor='white')
    ax.set_title(title, fontsize=11)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.set_xlabel('Age Group')

plt.suptitle(f'Fair Lending Analysis — Threshold {THRESHOLD}', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/fair_lending_check.png', dpi=150)
plt.show()

print('\nFair lending summary:')
print(fairness.to_string(index=False))
print('\nGroups with approval rate more than 10pp below the mean may warrant regulatory review.')

## Summary: What This Model Enables

| Decision | Without Model | With Model (p ≤ 15%) |
|---|---|---|
| Approval rule | Heuristic / manual | Data-driven, probability-based |
| Default rate | Unknown at approval time | ~8% (vs population base rate ~6.7%) |
| Approval rate | Undefined | ~72% |
| Fair lending monitoring | Reactive | Proactive, measurable |
| Threshold tuning | Not possible | Adjustable by risk appetite |

**The threshold is a business decision, not a modeling decision.** A growth-oriented team may accept a 20% predicted default rate to drive approval volume. A conservative team may hold at 10%. The model makes that tradeoff explicit and quantifiable — which is its primary value.